# 📚 *Medical Image Processing*
## 🧠 **Liver Segmentation Challenge**
### 🖥️ Testing Script

---

### 👩‍🔬 **Authors**
- **Danna Barrientos — s335116**
- **Sofía Escobar — s336895**
- **Felipe Medina — s335131**
- **Matteo Canal — s345814**

---


1. Connection with Drive

In [ ]:
# Mount Google Drive to access project files.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Install useful libraries and paths

In [68]:
# ============================================================
#  STANDARD LIBRARIES
# ============================================================
import os
import json
from datetime import datetime
import shutil

# ============================================================
#  NUMERICAL AND SCIENTIFIC COMPUTING
# ============================================================
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist

# ============================================================
#  IMAGE PROCESSING AND COMPUTER VISION
# ============================================================
import cv2
from PIL import Image
from skimage import morphology, measure, segmentation

# ============================================================
#  DATA VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt

# ============================================================
#  DEEP LEARNING (PyTorch & Torchvision)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

# ============================================================
#  UTILITIES
# ============================================================
from tqdm import tqdm

In [69]:
# =========================
#  Configuration & Path Setup (FINAL TEST INFERENCE)
# =========================

# ---- Project root (Google Drive) ----
PROJECT_ROOT = "/content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver"

# ---- Professor-provided TEST dataset ----
test_base_path   = os.path.join(PROJECT_ROOT, "test")
test_images_dir  = os.path.join(test_base_path, "image")  # adjust if needed

# ---- Final model folder (ALREADY EXISTS) ----
# Put here the EXACT existing folder that already contains:
#   - best_model.pt
#   - best_threshold.txt
#   - training_params.json
FINAL_MODEL_DIR = os.path.join(PROJECT_ROOT, "FINAL_MODEL")

best_model_path = os.path.join(FINAL_MODEL_DIR, "best_model.pt")
thr_file        = os.path.join(FINAL_MODEL_DIR, "best_threshold.txt")
params_path     = os.path.join(FINAL_MODEL_DIR, "training_params.json")

# ---- Outputs (inside the existing model folder) ----
output_masks_dir = os.path.join(FINAL_MODEL_DIR, "mask")
results_dir      = FINAL_MODEL_DIR
os.makedirs(output_masks_dir, exist_ok=True)  # safe: creates only /mask if missing

# =========================
#  Compatibility Variables (KEEP for downstream code)
# =========================
RUN_ID          = os.path.basename(FINAL_MODEL_DIR)
base_path_pre   = test_base_path
checkpoint_root = os.path.join(base_path_pre, "checkpoints")  # kept for compatibility
run_dir         = FINAL_MODEL_DIR

# =========================
#  Hardware Device Configuration
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
#  Path Validation (Sanity Checks)
# =========================
assert os.path.exists(PROJECT_ROOT),      f"PROJECT_ROOT not found: {PROJECT_ROOT}"
assert os.path.exists(test_base_path),    f"Test dataset folder not found: {test_base_path}"
assert os.path.exists(test_images_dir),   f"Test image directory not found: {test_images_dir}"
assert os.path.exists(FINAL_MODEL_DIR),   f"FINAL_MODEL_DIR not found: {FINAL_MODEL_DIR}"
assert os.path.exists(best_model_path),   f"Best model not found: {best_model_path}"

if not os.path.exists(thr_file):
    print(f"Warning: threshold file not found -> {thr_file} (OK if you use a fixed threshold)")

# =========================
#  Print Current Configuration
# =========================
print("PROJECT_ROOT      :", PROJECT_ROOT)
print("test_base_path    :", test_base_path)
print("test_images_dir   :", test_images_dir)
print("FINAL_MODEL_DIR   :", FINAL_MODEL_DIR)
print("best_model_path   :", best_model_path)
print("thr_file          :", thr_file)
print("params_path       :", params_path)
print("output_masks_dir  :", output_masks_dir)
print("results_dir       :", results_dir)

Device: cuda
PROJECT_ROOT      : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver
test_base_path    : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test
test_images_dir   : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test/image
FINAL_MODEL_DIR   : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL
best_model_path   : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/best_model.pt
thr_file          : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/best_threshold.txt
params_path       : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/training_params.json
output_masks_dir  : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/mask
results_dir       : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL


3. Preprocessing

In [70]:
# =========================
# PREPROCESSING CONFIG (FINAL TEST ONLY)
# =========================

# Input: professor test set
INPUT_TEST_DIR = test_images_dir  # comes from your path setup block

# Output: inside the EXISTING model folder (so nothing "new" is invented)
# This creates a processed test folder you can feed into inference.
OUTPUT_TEST_DIR = os.path.join(PROJECT_ROOT, "test_preprocessed", "image")
os.makedirs(OUTPUT_TEST_DIR, exist_ok=True)

# (Optional) keep a "dataset-like" base for downstream compatibility, if needed
# base_path_pre can remain the raw test_base_path from your setup.
test_images_dir_pre = OUTPUT_TEST_DIR  # useful if later code expects a dir to read from

# Target image (representative, tissue-rich) to compute the TARGET stain space.
# IMPORTANT: this MUST exist. Best practice: keep a fixed target from your training set,
# or store a chosen target image inside your model folder for reproducibility.
TARGET_IMG_PATH = "/content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/1001340_72.png"

# Softness: 0 -> original, 1 -> full Macenko
ALPHA_BLEND = 0.2

# Macenko parameters (safe)
OD_EPS = 1e-6
BETA = 0.15
ALPHA = 1.0
OD_CLIP = 2.5
C_CLIP = 5.0
BACKGROUND_V = 245

# =========================
# HELPERS
# =========================
def is_image_file(fname):
    f = fname.lower()
    return f.endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def rgb2od(I):
    I = I.astype(np.float32)
    I = np.clip(I, 1.0, 255.0)
    return -np.log((I + 1.0) / 256.0)

def od2rgb(OD):
    I = (np.exp(-OD) * 256.0) - 1.0
    return np.clip(I, 0, 255).astype(np.uint8)

def tissue_mask(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2]
    mask = (v < BACKGROUND_V).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask

def get_stain_matrix_macenko(img_rgb):
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    m = tissue_mask(img_rgb)
    OD_tissue = OD[m == 1].reshape(-1, 3)

    keep = np.linalg.norm(OD_tissue, axis=1) > BETA
    OD_tissue = OD_tissue[keep]
    if OD_tissue.shape[0] < 50:
        return None

    cov = np.cov(OD_tissue.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    V = eigvecs[:, np.argsort(eigvals)[::-1]]

    proj = OD_tissue @ V[:, :2]
    angles = np.arctan2(proj[:, 1], proj[:, 0])

    min_a = np.percentile(angles, ALPHA)
    max_a = np.percentile(angles, 100 - ALPHA)

    v1 = V[:, :2] @ np.array([np.cos(min_a), np.sin(min_a)])
    v2 = V[:, :2] @ np.array([np.cos(max_a), np.sin(max_a)])

    if v1[0] < v2[0]:
        W = np.stack([v2, v1], axis=1)
    else:
        W = np.stack([v1, v2], axis=1)

    W = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-8)
    return W

def get_concentrations(OD, W):
    pinv = np.linalg.pinv(W)
    C = OD @ pinv.T
    C = np.clip(C, 0, C_CLIP)
    return C

def macenko_soft_normalize(img_rgb, W_src, W_tgt, Ctgt_ref):
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    H, W = img_rgb.shape[:2]
    OD_flat = OD.reshape(-1, 3)

    C_src = get_concentrations(OD_flat, W_src)

    p = 99
    src_p = np.percentile(C_src, p, axis=0) + 1e-6
    tgt_p = Ctgt_ref + 1e-6
    scale = tgt_p / src_p

    C_adj = C_src * scale[None, :]
    C_adj = np.clip(C_adj, 0, C_CLIP)

    OD_rec = (C_adj @ W_tgt.T).reshape(H, W, 3)
    rgb_rec = od2rgb(OD_rec)

    out = (1 - ALPHA_BLEND) * img_rgb.astype(np.float32) + ALPHA_BLEND * rgb_rec.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)

# =========================
# PREPARE TARGET (fixed reference)
# =========================
assert os.path.exists(TARGET_IMG_PATH), f"❌ Target image not found: {TARGET_IMG_PATH}"
bgr = cv2.imread(TARGET_IMG_PATH)
rgb_tgt = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

W_tgt = get_stain_matrix_macenko(rgb_tgt)
assert W_tgt is not None, "❌ Could not estimate stain matrix for target. Pick another target."

OD_tgt = np.clip(rgb2od(rgb_tgt), 0, OD_CLIP).reshape(-1, 3)
C_tgt = get_concentrations(OD_tgt, W_tgt)
Ctgt_ref = np.percentile(C_tgt, 99, axis=0)

print("✅ Target stain matrix W_tgt:\n", W_tgt)
print("✅ Target concentration ref (p99):", Ctgt_ref)
print("✅ ALPHA_BLEND (softness):", ALPHA_BLEND)

# =========================
# MAIN (TEST ONLY)
# =========================
assert os.path.exists(INPUT_TEST_DIR), f"❌ Missing professor test dir: {INPUT_TEST_DIR}"
files = [f for f in os.listdir(INPUT_TEST_DIR) if is_image_file(f)]
assert len(files) > 0, f"❌ No images found in: {INPUT_TEST_DIR}"

print(f"\n=== FINAL TEST PREPROCESSING ===")
print(f"Input : {INPUT_TEST_DIR}")
print(f"Output: {OUTPUT_TEST_DIR}")
print(f"Processing {len(files)} test images...")

for fname in tqdm(sorted(files), ncols=80):
    in_path  = os.path.join(INPUT_TEST_DIR, fname)
    out_path = os.path.join(OUTPUT_TEST_DIR, fname)

    bgr = cv2.imread(in_path)
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    W_src = get_stain_matrix_macenko(rgb)
    if W_src is None:
        rgb_out = rgb  # safe fallback
    else:
        rgb_out = macenko_soft_normalize(rgb, W_src, W_tgt, Ctgt_ref)

    cv2.imwrite(out_path, cv2.cvtColor(rgb_out, cv2.COLOR_RGB2BGR))

print("\n✅ DONE. Preprocessed test set saved at:")
print(OUTPUT_TEST_DIR)

# =========================
# DOWNSTREAM HOOK (important)
# =========================
# If your inference code reads from "test_images_dir", redirect it here:
test_images_dir = OUTPUT_TEST_DIR

✅ Target stain matrix W_tgt:
 [[0.50340349 0.48482309]
 [0.76753112 0.7763269 ]
 [0.39683862 0.40281893]]
✅ Target concentration ref (p99): [1.79090213 5.        ]
✅ ALPHA_BLEND (softness): 0.2

=== FINAL TEST PREPROCESSING ===
Input : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test/image
Output: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test_preprocessed/image
Processing 1 test images...


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  6.88it/s]


✅ DONE. Preprocessed test set saved at:
/content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test_preprocessed/image


4. Model definition

In [71]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # checkpoint has NO conv biases -> bias=False
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_filters=32, depth=3, bilinear=True):
        super().__init__()
        self.depth = depth
        self.pool = nn.MaxPool2d(2)

        # names must match: down.*, up.*, bottleneck.*, out_conv.*
        self.down = nn.ModuleList()

        filters = init_filters
        c_in = in_channels
        for _ in range(depth):
            self.down.append(DoubleConv(c_in, filters))
            c_in = filters
            filters *= 2

        self.bottleneck = DoubleConv(c_in, filters)

        self.up = nn.ModuleList()
        for _ in range(depth):
            filters //= 2
            if bilinear:
                # checkpoint missing "up.0.up.1.bias" -> make this conv bias=False too
                up_layer = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1, bias=False),
                )
            else:
                up_layer = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)

            self.up.append(nn.ModuleDict({
                "up": up_layer,
                "conv": DoubleConv(filters * 2, filters),
            }))

        # checkpoint uses out_conv, not out
        self.out_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1, bias=True)

    def forward(self, x):
        skips = []
        for d in self.down:
            x = d(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for i in range(self.depth):
            skip = skips[-(i + 1)]
            x_up = self.up[i]["up"](x)

            if x_up.shape[-2:] != skip.shape[-2:]:
                x_up = F.interpolate(x_up, size=skip.shape[-2:], mode="bilinear", align_corners=False)

            x = torch.cat([skip, x_up], dim=1)
            x = self.up[i]["conv"](x)

        return self.out_conv(x)

5. Load checkpoint

In [72]:
def load_model_from_checkpoint(checkpoint_dir=None, epoch_to_load=None, use_best=True, strict=True, device=None):
    """
    Final-inference loader:
    - Reads params from 'params_path' if available (global), otherwise tries common filenames in checkpoint_dir.
    - Loads weights from 'best_model_path' if available (global), otherwise uses best_model.pt inside checkpoint_dir.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # =========================
    # Resolve checkpoint directory
    # =========================
    if checkpoint_dir is None:
        # Fallback to global run_dir / FINAL_MODEL_DIR if you defined it in the setup block
        if "run_dir" in globals():
            checkpoint_dir = run_dir
        elif "FINAL_MODEL_DIR" in globals():
            checkpoint_dir = FINAL_MODEL_DIR
        else:
            raise ValueError("checkpoint_dir is None and no global 'run_dir' / 'FINAL_MODEL_DIR' was found.")

    assert os.path.exists(checkpoint_dir), f"Checkpoint dir not found: {checkpoint_dir}"

    # =========================
    # Resolve params file
    # =========================
    candidate_params = []

    # Prefer the path you already define in your setup block (training_params.json)
    if "params_path" in globals():
        candidate_params.append(params_path)

    # Common fallbacks
    candidate_params += [
        os.path.join(checkpoint_dir, "training_params.json"),
        os.path.join(checkpoint_dir, "run_config.json"),
        os.path.join(checkpoint_dir, "training_config.json"),
        os.path.join(checkpoint_dir, "config.json"),
    ]

    params_file = next((p for p in candidate_params if os.path.exists(p)), None)
    assert params_file is not None, f"No params json found. Tried: {candidate_params}"

    with open(params_file, "r") as f:
        params = json.load(f)

    # =========================
    # Model initialization (from params)
    # =========================
    model = UNet(
        in_channels=params.get("in_channels", 3),
        out_channels=params.get("out_channels", 1),
        init_filters=params.get("init_filters", 32),
        depth=params.get("depth", 3),
        bilinear=params.get("bilinear_up", True),
    ).to(device)

    # =========================
    # Resolve checkpoint weights file
    # =========================
    candidate_ckpts = []

    # Prefer the explicit best_model_path you already define in your setup block
    if "best_model_path" in globals():
        candidate_ckpts.append(best_model_path)

    # Common fallbacks inside dir
    best_path = os.path.join(checkpoint_dir, "best_model.pt")
    candidate_ckpts.append(best_path)

    if not use_best:
        assert epoch_to_load is not None, "epoch_to_load is required when use_best=False"
        candidate_ckpts.append(os.path.join(checkpoint_dir, f"model_epoch_{epoch_to_load}.pt"))

    ckpt_path = next((p for p in candidate_ckpts if os.path.exists(p)), None)
    assert ckpt_path is not None, f"No checkpoint found. Tried: {candidate_ckpts}"

    # =========================
    # Load weights
    # =========================
    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    if strict:
        model.load_state_dict(state, strict=True)
        missing, unexpected = [], []
    else:
        missing, unexpected = model.load_state_dict(state, strict=False)

    model.eval()

    # =========================
    # Input size (robust)
    # =========================
    input_size = tuple(params.get("input_size", params.get("img_size", (512, 512))))
    if isinstance(input_size, int):
        input_size = (input_size, input_size)

    print("Loaded checkpoint:", ckpt_path)
    print("Loaded params from:", params_file)
    print("Input size:", input_size)
    if not strict and (missing or unexpected):
        print("Missing keys:", missing)
        print("Unexpected keys:", unexpected)

    return model, params, input_size, device

In [73]:
# =========================
#  Load FINAL model + threshold (FINAL TEST INFERENCE)
# =========================

# Use the existing final model folder (already created)
checkpoint_dir = FINAL_MODEL_DIR  # or run_dir (both point to the same)

assert os.path.exists(checkpoint_dir), f"Missing checkpoint_dir: {checkpoint_dir}"

# -------------------------
# Load run configuration parameters
# -------------------------
# Prefer the explicit params_path from your setup block.
candidate_configs = []
if "params_path" in globals():
    candidate_configs.append(params_path)

# Common fallbacks (in case the filename differs)
candidate_configs += [
    os.path.join(checkpoint_dir, "training_params.json"),
    os.path.join(checkpoint_dir, "run_config.json"),
    os.path.join(checkpoint_dir, "training_config.json"),
    os.path.join(checkpoint_dir, "config.json"),
]

config_path = next((p for p in candidate_configs if os.path.exists(p)), None)
assert config_path is not None, f"No config json found. Tried: {candidate_configs}"

with open(config_path, "r") as f:
    params = json.load(f)

# Extract input image size from configuration (robust)
input_size = params.get("input_size", params.get("img_size", (512, 512)))
if isinstance(input_size, int):
    input_size = (input_size, input_size)
input_size = tuple(input_size)

# -------------------------
# Initialize model
# -------------------------
model = UNet(
    in_channels=params.get("in_channels", 3),
    out_channels=params.get("out_channels", 1),
    init_filters=params.get("init_filters", 32),
    depth=params.get("depth", 3),
    bilinear=params.get("bilinear_up", True),
).to(device)

# -------------------------
# Load best model weights
# -------------------------
# Prefer explicit best_model_path from setup block.
candidate_ckpts = []
if "best_model_path" in globals():
    candidate_ckpts.append(best_model_path)
candidate_ckpts.append(os.path.join(checkpoint_dir, "best_model.pt"))

best_ckpt = next((p for p in candidate_ckpts if os.path.exists(p)), None)
assert best_ckpt is not None, f"Best model not found. Tried: {candidate_ckpts}"

state = torch.load(best_ckpt, map_location=device)
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]

model.load_state_dict(state, strict=True)
model.eval()

# -------------------------
# Load best threshold
# -------------------------
BEST_THR = 0.35  # fallback
_thr_path = thr_file if "thr_file" in globals() else os.path.join(checkpoint_dir, "best_threshold.txt")
if os.path.exists(_thr_path):
    with open(_thr_path, "r") as f:
        BEST_THR = float(f.readline().strip())

print("Model loaded correctly")
print("checkpoint_dir:", checkpoint_dir)
print("config_path   :", config_path)
print("weights       :", best_ckpt)
print("input_size    :", input_size)
print("BEST_THR      :", BEST_THR)


Model loaded correctly
checkpoint_dir: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL
config_path   : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/training_params.json
weights       : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/best_model.pt
input_size    : (416, 416)
BEST_THR      : 0.35


6. Apply trained model to the test set




In [74]:
# =========================================================
# BLOCK A) TEST INFERENCE (POSTPROCESS HAPPENS HERE)
# =========================================================

# =========================
# POSTPROCESS CONFIG
# =========================
USE_POSTPROC   = True
POSTPROC_MODE  = "min_area_fill"
RG_SEED_THR    = 0.80
RG_GROW_THR    = 0.50
MIN_OBJ_AREA   = 10

# =========================
# EXPECTED GLOBALS (from setup / model-loading cells)
# =========================
# device, model, test_images_dir, output_masks_dir, input_size, BEST_THR
# USE_POSTPROC, POSTPROC_MODE, MIN_OBJ_AREA
# RG_SEED_THR, RG_GROW_THR (only if POSTPROC_MODE == "region_growing")

# =========================
# Preprocessing (match training)
# =========================
# Make input_size robust (int -> (int,int), list -> tuple)
if isinstance(input_size, int):
    _resize_size = (input_size, input_size)
else:
    _resize_size = tuple(input_size)

preprocess = transforms.Compose([
    transforms.Resize(_resize_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

# =========================
# Postprocess (MODULAR)
# =========================
def postprocess_probs_to_mask(probs_up, thr):
    """
    probs_up: float32 [0,1] (original image size)
    thr: float threshold (BEST_THR)
    returns: uint8 mask {0,1}
    """
    mask_bin = (probs_up >= thr).astype(np.uint8)

    if (not USE_POSTPROC) or (POSTPROC_MODE == "none"):
        return mask_bin

    if POSTPROC_MODE == "min_area":
        out = morphology.remove_small_objects(mask_bin.astype(bool), MIN_OBJ_AREA)
        return out.astype(np.uint8)

    if POSTPROC_MODE == "min_area_fill":
        out = morphology.remove_small_objects(mask_bin.astype(bool), MIN_OBJ_AREA)
        out = ndi.binary_fill_holes(out)
        return out.astype(np.uint8)

    if POSTPROC_MODE == "watershed":
        base = morphology.remove_small_objects(mask_bin.astype(bool), MIN_OBJ_AREA)
        if base.sum() == 0:
            return mask_bin
        dist = ndi.distance_transform_edt(base)
        peaks = morphology.local_maxima(dist)
        markers = measure.label(peaks)
        labels = segmentation.watershed(-dist, markers, mask=base)
        return (labels > 0).astype(np.uint8)

    if POSTPROC_MODE == "region_growing":
        seeds = (probs_up >= RG_SEED_THR)
        grow  = (probs_up >= RG_GROW_THR)

        grow_lab = measure.label(grow)
        if grow_lab.max() == 0:
            out = np.zeros_like(mask_bin, dtype=np.uint8)
        else:
            seed_ids = np.unique(grow_lab[seeds])
            seed_ids = seed_ids[seed_ids != 0]
            out = np.isin(grow_lab, seed_ids).astype(np.uint8)

        out = morphology.remove_small_objects(out.astype(bool), MIN_OBJ_AREA).astype(np.uint8)
        out = ndi.binary_fill_holes(out).astype(np.uint8)
        return out

    raise ValueError(f"Unknown POSTPROC_MODE: {POSTPROC_MODE}")

# =========================
# Inference Loop
# =========================
os.makedirs(output_masks_dir, exist_ok=True)

valid_ext = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
image_files = sorted([f for f in os.listdir(test_images_dir) if f.lower().endswith(valid_ext)])

assert len(image_files) > 0, f"No test images found in: {test_images_dir}"

print(f"Found {len(image_files)} test images. Running inference...")
print("Checkpoint dir   :", FINAL_MODEL_DIR if "FINAL_MODEL_DIR" in globals() else "N/A")
print("Test images dir  :", test_images_dir)
print("Output masks dir :", output_masks_dir)
print("Resize input_size:", _resize_size)
print("BEST_THR used    :", BEST_THR)
print("POSTPROC         :", USE_POSTPROC, "| MODE:", POSTPROC_MODE, "| MIN_OBJ_AREA:", MIN_OBJ_AREA)
if USE_POSTPROC and POSTPROC_MODE == "region_growing":
    print("RG params        :", "SEED=", RG_SEED_THR, "GROW=", RG_GROW_THR)
print()

model.eval()

for img_name in tqdm(image_files, desc="Generating masks", ncols=80):
    img_path = os.path.join(test_images_dir, img_name)

    image = Image.open(img_path).convert("RGB")
    W, H = image.size

    x = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs  = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

    # Upsample back to original resolution
    probs_up = cv2.resize(probs, (W, H), interpolation=cv2.INTER_LINEAR)

    # Final mask
    mask_final = postprocess_probs_to_mask(probs_up, thr=BEST_THR)  # {0,1}
    mask_u8 = (mask_final * 255).astype(np.uint8)

    out_path = os.path.join(output_masks_dir, img_name)
    Image.fromarray(mask_u8, mode="L").save(out_path)

print("\n✅ Inference completed.")
print("Saved masks to:", output_masks_dir)

Found 1 test images. Running inference...
Checkpoint dir   : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL
Test images dir  : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test_preprocessed/image
Output masks dir : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/mask
Resize input_size: (416, 416)
BEST_THR used    : 0.35
POSTPROC         : True | MODE: min_area_fill | MIN_OBJ_AREA: 10



Generating masks:   0%|                                   | 0/1 [00:00<?, ?it/s]/tmp/ipython-input-31152635.py:131: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(mask_u8, mode="L").save(out_path)
Generating masks: 100%|███████████████████████████| 1/1 [00:00<00:00, 17.06it/s]


✅ Inference completed.
Saved masks to: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/mask


7. Evaluation

In [75]:
# =========================================================
# BLOCK B) EVALUATION (MEASURES SAVED MASKS) \u2014 PROFESSOR TEST WITH GT
# Assumes professor test set structure:
#   test_base_path/
#       test/
#           image/
#           manual_py/   (optional)
#           manual/      (fallback)
# - Evaluates masks saved in output_masks_dir (pred_dir)
# - Uses GT from manual_py if exists else manual
# - Does NOT re-apply postproc (already baked into saved masks)
# =========================================================

# =========================
# Directories
# =========================
pred_dir = output_masks_dir

# Corrected ground truth directory paths
gt_dir_py = os.path.join(test_base_path, "manual_py")
gt_dir    = gt_dir_py if os.path.exists(gt_dir_py) else os.path.join(test_base_path, "manual")

assert os.path.exists(pred_dir), f"\u274c pred_dir not found: {pred_dir}"
assert os.path.exists(gt_dir),   f"\u274c gt_dir not found: {gt_dir}"

# =========================
# Output files
# =========================
output_txt  = os.path.join(results_dir, "metrics_summary.txt")
output_json = os.path.join(results_dir, "metrics_summary.json")
output_cfg  = os.path.join(results_dir, "eval_config.json")
output_csv  = os.path.join(results_dir, "per_image_metrics.csv")

IOU_OBJ_THRESH = 0.5
ASSUME_UINT8 = True

# =========================
# Helpers
# =========================
def load_mask_binary(path, assume_uint8=True):
    m = np.array(Image.open(path))
    if m.ndim == 3:
        m = m[..., 0]
    m = m.astype(np.float32)

    # Common case: 0/255 masks
    if assume_uint8 and m.max() > 1.0:
        return (m > 127).astype(np.uint8)

    # Case: already 0/1 (or [0,1])
    return (m >= 0.5).astype(np.uint8)

def dice_or_skip(gt, pr):
    s_gt, s_pr = int(gt.sum()), int(pr.sum())
    if s_gt == 0 and s_pr == 0:
        return None
    inter = np.sum(gt * pr)
    return (2.0 * inter) / (s_gt + s_pr + 1e-8)

def iou_coef(gt, pr):
    inter = np.sum(gt * pr)
    union = gt.sum() + pr.sum() - inter
    return inter / (union + 1e-8)

def pixel_precision_recall(gt, pr):
    tp = np.sum((gt == 1) & (pr == 1))
    fp = np.sum((gt == 0) & (pr == 1))
    fn = np.sum((gt == 1) & (pr == 0))
    return tp / (tp + fp + 1e-8), tp / (tp + fn + 1e-8)

def count_vacuoles(mask):
    _, n = ndi.label(mask)
    return int(n)

def steatosis_percentage(mask):
    return (mask.sum() / mask.size) * 100.0

def object_f1(gt, pr, iou_thresh=0.5):
    gt_lab, gt_n = ndi.label(gt)
    pr_lab, pr_n = ndi.label(pr)
    gt_n, pr_n = int(gt_n), int(pr_n)

    if gt_n == 0 and pr_n == 0:
        return 1.0
    if gt_n == 0 or pr_n == 0:
        return 0.0

    iou_mat = np.zeros((gt_n, pr_n), dtype=np.float32)
    for i in range(1, gt_n + 1):
        g = (gt_lab == i)
        g_area = g.sum()
        if g_area == 0:
            continue
        for j in range(1, pr_n + 1):
            p = (pr_lab == j)
            inter = np.logical_and(g, p).sum()
            if inter == 0:
                continue
            union = g_area + p.sum() - inter
            iou_mat[i-1, j-1] = inter / (union + 1e-8)

    gt_match = (iou_mat.max(axis=1) >= iou_thresh)
    pr_match = (iou_mat.max(axis=0) >= iou_thresh)

    tp = int(gt_match.sum())
    fn = gt_n - tp
    fp = pr_n - int(pr_match.sum())

    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    return 2 * prec * rec / (prec + rec + 1e-8)

def mean_std(x):
    return float(np.mean(x)), float(np.std(x))

# =========================
# File intersection (pred vs GT)
# =========================
valid_ext = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')
pred_files = sorted([f for f in os.listdir(pred_dir) if f.lower().endswith(valid_ext)])
gt_files   = set([f for f in os.listdir(gt_dir)   if f.lower().endswith(valid_ext)])

image_files = [f for f in pred_files if f in gt_files]
missing_gt  = [f for f in pred_files if f not in gt_files]
missing_pr  = [f for f in gt_files   if f not in set(pred_files)]

assert len(image_files) > 0, (
    "\u274c No matching filenames between pred and GT.\n"
    f"Example pred files (first 10): {pred_files[:10]}\n"
    f"Example gt files   (first 10): {sorted(list(gt_files))[:10]}\n"
)

# =========================
# Print config
# =========================
print("\n==================== EVALUATION CONFIG ====================")
print("RUN_ID               :", RUN_ID)
print("results_dir          :", results_dir)
print("pred_dir             :", pred_dir)
print("gt_dir               :", gt_dir)
print("test_base_path       :", test_base_path)
print("POSTPROC (inference) :", USE_POSTPROC, "| MODE:", POSTPROC_MODE, "| MIN_OBJ_AREA:", MIN_OBJ_AREA)
if USE_POSTPROC and POSTPROC_MODE == "region_growing":
    print("RG params:", "SEED=", RG_SEED_THR, "GROW=", RG_GROW_THR)
print("matched_files        :", len(image_files))
print("missing_gt_for_pred  :", len(missing_gt))
print("missing_pred_for_gt  :", len(missing_pr))
print("===========================================================\n")

# =========================
# Metrics loop
# =========================
dice_list, iou_list, prec_list, rec_list = [], [], [], []
cnt_list, sp_list, objf1_list = [], [], []
skipped = 0
rows = []

for fname in tqdm(image_files, desc="Evaluating", ncols=80):
    gt = load_mask_binary(os.path.join(gt_dir, fname), assume_uint8=ASSUME_UINT8)
    pr = load_mask_binary(os.path.join(pred_dir, fname), assume_uint8=ASSUME_UINT8)

    d = dice_or_skip(gt, pr)
    if d is None:
        skipped += 1
    else:
        dice_list.append(d)

    iou = iou_coef(gt, pr)
    p, r = pixel_precision_recall(gt, pr)
    cnt_err = abs(count_vacuoles(gt) - count_vacuoles(pr))
    sp_err  = abs(steatosis_percentage(gt) - steatosis_percentage(pr))
    objf1   = object_f1(gt, pr, IOU_OBJ_THRESH)

    iou_list.append(iou)
    prec_list.append(p)
    rec_list.append(r)
    cnt_list.append(cnt_err)
    sp_list.append(sp_err)
    objf1_list.append(objf1)

    rows.append({
        "file": fname,
        "dice": np.nan if d is None else float(d),
        "iou": float(iou),
        "precision": float(p),
        "recall": float(r),
        "count_err": float(cnt_err),
        "steat_err": float(sp_err),
        "obj_f1": float(objf1),
        "skipped_empty_empty": int(d is None),
    })

summary = {
    "run_id": str(RUN_ID),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "test_base_path": str(test_base_path),
    "pred_dir": str(pred_dir),
    "gt_dir": str(gt_dir),
    "use_postproc": bool(USE_POSTPROC),
    "postproc_mode": str(POSTPROC_MODE),
    "min_obj_area": int(MIN_OBJ_AREA),
    "rg_seed_thr": float(RG_SEED_THR),
    "rg_grow_thr": float(RG_GROW_THR),
    "matched_images": int(len(image_files)),
    "missing_gt_for_pred": int(len(missing_gt)),
    "missing_pred_for_gt": int(len(missing_pr)),
    "dice_used": int(len(dice_list)),
    "dice_skipped_empty_empty": int(skipped),
}

summary["dice_mean"], summary["dice_std"] = mean_std(dice_list) if len(dice_list) else (float("nan"), float("nan"))
summary["iou_mean"],  summary["iou_std"]  = mean_std(iou_list)
summary["precision_mean"], summary["precision_std"] = mean_std(prec_list)
summary["recall_mean"],    summary["recall_std"]    = mean_std(rec_list)
summary["count_err_mean"], summary["count_err_std"] = mean_std(cnt_list)
summary["steat_err_mean"], summary["steat_err_std"] = mean_std(sp_list)
summary["obj_f1_mean"],    summary["obj_f1_std"]    = mean_std(objf1_list)

# =========================
# Save outputs
# =========================
with open(output_json, "w") as f:
    json.dump(summary, f, indent=2)

pd.DataFrame(rows).to_csv(output_csv, index=False)

with open(output_txt, "w") as f:
    f.write(json.dumps(summary, indent=2))

with open(output_cfg, "w") as f:
    json.dump(summary, f, indent=2)

# =========================
# Print summary
# =========================
print("\n==================== FINAL RESULTS ====================")
print(f"RUN_ID: {RUN_ID} | matched={len(image_files)}")
print(f"Dice (skip empty-empty): {summary['dice_mean']:.4f} \u00b1 {summary['dice_std']:.4f} | used={len(dice_list)} skipped={skipped}")
print(f"IoU                    : {summary['iou_mean']:.4f} \u00b1 {summary['iou_std']:.4f}")
print(f"Precision              : {summary['precision_mean']:.4f} \u00b1 {summary['precision_std']:.4f}")
print(f"Recall                 : {summary['recall_mean']:.4f} \u00b1 {summary['recall_std']:.4f}")
print(f"Count error            : {summary['count_err_mean']:.2f} \u00b1 {summary['count_err_std']:.2f}")
print(f"Steatosis % error      : {summary['steat_err_mean']:.2f}% \u00b1 {summary['steat_err_std']:.2f}%")
print(f"Object-level F1        : {summary['obj_f1_mean']:.4f} \u00b1 {summary['obj_f1_std']:.4f}")
print("=======================================================\n")

print("\u2705 Saved to:", results_dir)
print(" -", output_txt)
print(" -", output_json)
print(" -", output_cfg)
print(" -", output_csv)



==================== EVALUATION CONFIG ====================
RUN_ID               : FINAL_MODEL
results_dir          : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL
pred_dir             : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/mask
gt_dir               : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test/manual_py
test_base_path       : /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/test
POSTPROC (inference) : True | MODE: min_area_fill | MIN_OBJ_AREA: 10
matched_files        : 1
missing_gt_for_pred  : 0
missing_pred_for_gt  : 0



Evaluating: 100%|█████████████████████████████████| 1/1 [00:01<00:00,  1.32s/it]


==================== FINAL RESULTS ====================
RUN_ID: FINAL_MODEL | matched=1
Dice (skip empty-empty): 0.7854 ± 0.0000 | used=1 skipped=0
IoU                    : 0.6466 ± 0.0000
Precision              : 0.8552 ± 0.0000
Recall                 : 0.7261 ± 0.0000
Count error            : 12.00 ± 0.00
Steatosis % error      : 2.29% ± 0.00%
Object-level F1        : 0.6543 ± 0.0000

✅ Saved to: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL
 - /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/metrics_summary.txt
 - /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/metrics_summary.json
 - /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/eval_config.json
 - /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/per_image_metrics.csv


8. Debug

In [76]:
# =========================================================
# DEBUG COMPARISONS
# - Generates and saves visual comparisons between original image, GT, and predicted masks.
# - Outputs: panel images and overlay images for qualitative analysis.
# =========================================================

# =========================
# Path Configuration
# =========================
# Image directory (consistent with the PATHS cell).
img_dir  = os.path.join(base_path_pre, "image")

# Ground truth directory, prioritizing 'manual_py' if available.
# Corrected ground truth directory paths
gt_dir_py = os.path.join(base_path_pre, "manual_py")
gt_dir    = gt_dir_py if os.path.exists(gt_dir_py) else os.path.join(base_path_pre, "manual")

# Predicted masks directory.
pred_dir  = output_masks_dir

# Directory to save debugging outputs; created if it does not exist.
debug_dir = os.path.join(results_dir, "debug_comparisons")
os.makedirs(debug_dir, exist_ok=True)

# Assert existence of necessary directories.
assert os.path.exists(img_dir),  f"img_dir not found: {img_dir}"
assert os.path.exists(gt_dir),   f"gt_dir not found: {gt_dir}"
assert os.path.exists(pred_dir), f"pred_dir not found: {pred_dir}"

# =========================
# Segmentation Threshold
# =========================
# Load the optimal threshold from file or use a fallback value.
thr_file = os.path.join(checkpoint_dir, "best_threshold.txt")
THRESH_FALLBACK = 0.35

if "BEST_THR" in globals():
    THR = float(BEST_THR)
    thr_source = "BEST_THR (already loaded)"
elif os.path.exists(thr_file):
    with open(thr_file, "r") as f:
        THR = float(f.readline().strip())
    thr_source = "best_threshold.txt"
else:
    THR = THRESH_FALLBACK
    thr_source = f"fallback({THRESH_FALLBACK})"

print("Debugging results_dir:", results_dir)
print(f" Using THR = {THR:.4f}  (source: {thr_source})")
print("debug_dir:", debug_dir)

# =========================
# Helper Functions
# =========================
def load_bin_mask(path, thr, assume_uint8=True):
    # Loads a mask image and binarizes it based on a threshold.
    m = np.array(Image.open(path))
    if m.ndim == 3:
        m = m[..., 0] # Extract single channel if RGB.
    m = m.astype(np.float32)
    if assume_uint8 and m.max() > 1.0:
        m /= 255.0 # Normalize 8-bit image to [0, 1].
    return (m >= thr).astype(np.uint8) # Apply threshold to create binary mask.

def overlay_masks_on_image(img_rgb, gt_bin, pr_bin, alpha=0.45):
    # Overlays ground truth (green) and predicted (red) masks onto the original image.
    base = img_rgb.astype(np.float32) / 255.0 # Normalize image to [0, 1].
    overlay = base.copy()

    # Color ground truth (GT) regions green.
    overlay[gt_bin == 1, 1] = 1.0
    overlay[gt_bin == 1, 0] *= 0.2
    overlay[gt_bin == 1, 2] *= 0.2

    # Color predicted (Pred) regions red.
    overlay[pr_bin == 1, 0] = 1.0
    overlay[pr_bin == 1, 1] *= 0.2
    overlay[pr_bin == 1, 2] *= 0.2

    # Blend original image with overlay.
    out = (1 - alpha) * base + alpha * overlay
    return (np.clip(out, 0, 1) * 255).astype(np.uint8)

# =========================
# Identify Common Files for Debugging
# =========================
# Find images that exist in all three directories (original, GT, predicted).
valid_ext = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')
img_files  = {f for f in os.listdir(img_dir)  if f.lower().endswith(valid_ext)}
gt_files   = {f for f in os.listdir(gt_dir)   if f.lower().endswith(valid_ext)}
pred_files = {f for f in os.listdir(pred_dir) if f.lower().endswith(valid_ext)}

common_files = sorted(list(img_files & gt_files & pred_files))
print(f"Found {len(common_files)} images to debug.")

assert len(common_files) > 0, (
    "No common files among image/GT/pred.\n"
    f"Example img files (first 10): {sorted(list(img_files))[:10]}\n"
    f"Example gt files  (first 10): {sorted(list(gt_files))[:10]}\n"
    f"Example pred files(first 10): {sorted(list(pred_files))[:10]}\n"
)

# =========================
# Debugging Loop
# =========================
# Process each common file to generate and save comparison visuals.
for fname in tqdm(common_files, desc="Saving debug panels", ncols=80):
    img_path  = os.path.join(img_dir, fname)
    gt_path   = os.path.join(gt_dir, fname)
    pred_path = os.path.join(pred_dir, fname)

    # Load original image.
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    # Load and binarize ground truth and predicted masks.
    gt_bin   = load_bin_mask(gt_path,   THR)
    pred_bin = load_bin_mask(pred_path, THR)

    # Resize predicted mask to match ground truth dimensions if necessary.
    if pred_bin.shape != gt_bin.shape:
        pred_bin = np.array(
            Image.fromarray((pred_bin * 255).astype(np.uint8)).resize(
                (gt_bin.shape[1], gt_bin.shape[0]),
                resample=Image.NEAREST
            )
        )
        pred_bin = (pred_bin > 127).astype(np.uint8) # Re-binarize after resize.

    # Calculate True Positives (TP), False Positives (FP), and False Negatives (FN).
    tp = (gt_bin == 1) & (pred_bin == 1)
    fp = (gt_bin == 0) & (pred_bin == 1)
    fn = (gt_bin == 1) & (pred_bin == 0)

    # Create a visualization of errors: TP (white), FP (red), FN (blue).
    err_vis = np.zeros((*gt_bin.shape, 3), dtype=np.uint8)
    err_vis[tp] = [255, 255, 255] # White for true positives.
    err_vis[fp] = [255, 0, 0]      # Red for false positives.
    err_vis[fn] = [0, 0, 255]      # Blue for false negatives.

    # Generate an overlay image with colored masks.
    overlay = overlay_masks_on_image(img_np, gt_bin, pred_bin, alpha=0.45)

    # Create a four-panel figure for comparison.
    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(img_np); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(gt_bin, cmap="gray"); ax[1].set_title("GT (bin)"); ax[1].axis("off")
    ax[2].imshow(pred_bin, cmap="gray"); ax[2].set_title("Pred (bin)"); ax[2].axis("off")
    ax[3].imshow(err_vis); ax[3].set_title("Errors: FP red | FN blue"); ax[3].axis("off")
    plt.tight_layout()

    # Define output file paths for panel and overlay images.
    base = os.path.splitext(fname)[0]
    panel_path   = os.path.join(debug_dir, f"{base}_panel.png")
    overlay_path = os.path.join(debug_dir, f"{base}_overlay.png")

    # Save the panel figure and the overlay image.
    plt.savefig(panel_path, dpi=150, bbox_inches="tight")
    plt.close() # Close plot to prevent memory issues.
    Image.fromarray(overlay).save(overlay_path)

print("\n All debug comparison images saved!")
print("Folder:", debug_dir)


Debugging results_dir: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL
 Using THR = 0.3500  (source: BEST_THR (already loaded))
debug_dir: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/debug_comparisons
Found 1 images to debug.


Saving debug panels: 100%|████████████████████████| 1/1 [00:00<00:00,  1.36it/s]


 All debug comparison images saved!
Folder: /content/drive/MyDrive/Challenge_Liver_MIP/Group_AS11_Challenge_Liver/FINAL_MODEL/debug_comparisons
